## Structured Tools with Pydantic

In the earlier notebooks we defined tools using plain Python type hints.
For real-world tools that take many parameters, have optional fields, or need
validation logic, we use **Pydantic `BaseModel`** to define a full input schema.

This matters because **the LLM reads the schema** to decide what arguments to fill in.
Richer field descriptions → more reliable tool calls.

**Topics covered:**
* Simple `@tool` recap and schema inspection
* `BaseModel` + `Field(description=...)` for richer schemas
* Optional parameters with defaults
* Overriding tool `name` and `description`
* Using structured tools inside an agent

In [ ]:
%run langchain_common.py

## 1. Simple `@tool` — what schema does the LLM actually see?

Before adding Pydantic, let's inspect the JSON schema that a plain tool exposes.
The only description the LLM gets is the function docstring.

In [ ]:
from langchain_core.tools import tool

@tool
def simple_add(a: int, b: int) -> int:
    """Add two integers together."""
    return a + b

# This is the JSON schema the LLM receives at inference time
pp(simple_add.args_schema.model_json_schema())

Notice: only the types are communicated — no hints about what `a` or `b` mean.
For a tool like `add`, this is fine. For a `search_flights` tool with 5 parameters
it can lead to the LLM guessing wrong values.

## 2. `BaseModel` + `Field` — richer per-field descriptions

In [ ]:
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city: str = Field(description="City name to get weather for, e.g. 'Lahore' or 'London'")
    unit: str = Field(default="celsius", description="Temperature unit: 'celsius' or 'fahrenheit'")

@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius") -> str:
    """Get the current weather for a city."""
    # Simulated data — replace with a real API call in production
    temps = {"lahore": 38, "london": 15, "new york": 22, "dubai": 42}
    temp_c = temps.get(city.lower(), 20)
    temp = temp_c if unit == "celsius" else round(temp_c * 9 / 5 + 32, 1)
    symbol = "C" if unit == "celsius" else "F"
    return f"Current temperature in {city}: {temp}°{symbol}"

# Now each field has a description the LLM can read
pp(get_weather.args_schema.model_json_schema())

In [ ]:
# Test the tool directly before wiring it into an agent
print(get_weather.invoke({"city": "Lahore"}))
print(get_weather.invoke({"city": "London", "unit": "fahrenheit"}))

## 3. Multi-field structured tool

Here's a more realistic example: a flight search tool with four parameters,
each with a clear description so the LLM knows exactly what format to use.

In [ ]:
class FlightSearchInput(BaseModel):
    origin: str = Field(description="IATA airport code for departure, e.g. 'LHE' for Lahore")
    destination: str = Field(description="IATA airport code for arrival, e.g. 'DXB' for Dubai")
    date: str = Field(description="Travel date in YYYY-MM-DD format")
    passengers: int = Field(default=1, description="Number of passengers (1–9)")

@tool(args_schema=FlightSearchInput)
def search_flights(origin: str, destination: str, date: str, passengers: int = 1) -> str:
    """Search for available flights between two airports on a given date."""
    return (
        f"Found 3 flights from {origin} → {destination} on {date} "
        f"for {passengers} passenger(s). Prices from $150."
    )

pp(search_flights.args_schema.model_json_schema())

In [ ]:
# Invoke directly — all args explicit
print(search_flights.invoke({"origin": "LHE", "destination": "DXB", "date": "2025-08-15", "passengers": 2}))

# Invoke with only required args — optional 'passengers' uses its default
print(search_flights.invoke({"origin": "LHE", "destination": "LHR", "date": "2025-09-01"}))

## 4. Overriding tool `name` and `description`

The `name` is what the LLM uses to select the tool. The `description` tells it *when* to use it.
You can override both independently of the Python function name.

In [ ]:
@tool(
    "currency_converter",
    description="Convert a monetary amount from one currency to another. Use this whenever a user asks about exchange rates or currency conversion."
)
def _convert(amount: float, from_currency: str, to_currency: str) -> str:
    """Internal conversion logic."""
    # Simulated fixed rates for demo purposes
    rates = {"USD": 1.0, "PKR": 280.0, "EUR": 0.92, "GBP": 0.79, "AED": 3.67}
    from_rate = rates.get(from_currency.upper(), 1.0)
    to_rate = rates.get(to_currency.upper(), 1.0)
    result = (amount / from_rate) * to_rate
    return f"{amount} {from_currency.upper()} = {result:.2f} {to_currency.upper()}"

print(f"Tool name        : {_convert.name}")
print(f"Tool description : {_convert.description}")
print()
print(_convert.invoke({"amount": 100, "from_currency": "USD", "to_currency": "PKR"}))
print(_convert.invoke({"amount": 500, "from_currency": "EUR", "to_currency": "GBP"}))

## 5. Using structured tools inside an agent

Now wire all three tools into a travel agent. Because each tool has detailed
schemas, the LLM can extract values confidently from natural-language queries.

In [ ]:
tools = [get_weather, search_flights, _convert]
travel_agent = create_agent(llm_noreason, tools)

queries = [
    "What is the weather in Dubai in fahrenheit?",
    "Find 2 flights from Lahore to Dubai on 2025-08-15.",
    "How much is 250 USD in Pakistani rupees?",
]

for query in queries:
    result = travel_agent.invoke({"messages": [HumanMessage(content=query)]})
    print(f"Q: {query}")
    print(f"A: {result['messages'][-1].content}")
    print("-" * 60)

## 6. Comparing schemas: plain `@tool` vs `BaseModel`

Side-by-side view of what changes when you add Pydantic.

In [ ]:
import json

# Plain tool — no per-field descriptions
@tool
def plain_book_hotel(city: str, check_in: str, nights: int) -> str:
    """Book a hotel room."""
    return f"Booked {nights} nights in {city} from {check_in}"

# Structured tool — every field described
class HotelInput(BaseModel):
    city: str = Field(description="City where the hotel is located, e.g. 'Istanbul'")
    check_in: str = Field(description="Check-in date in YYYY-MM-DD format")
    nights: int = Field(default=1, description="Number of nights to stay (1–30)")

@tool(args_schema=HotelInput)
def structured_book_hotel(city: str, check_in: str, nights: int = 1) -> str:
    """Book a hotel room."""
    return f"Booked {nights} nights in {city} from {check_in}"

print("=== PLAIN TOOL SCHEMA ===")
print(json.dumps(plain_book_hotel.args_schema.model_json_schema(), indent=2))
print()
print("=== STRUCTURED TOOL SCHEMA ===")
print(json.dumps(structured_book_hotel.args_schema.model_json_schema(), indent=2))